# Sprint 2 · Motor de inferencia v1 (Hawkes topológico) y resultados por fold

**Tesis:** Diagnóstico de causalidad de fallas en una red telco usando grafos · Región **Pasco**, Perú.

El motor ordena los equipos de cada incidente de la causa más probable a la menos probable (Top-k). Combina **tiempo**
(qué alarma llegó antes), **topología** (quién está aguas arriba y a cuántos saltos) y lo que **aprende de los datos**
(qué tipo de alarma suele disparar a cuál), sin usar etiquetas ni las reglas de los manuales. Código en `src/motor.py`;
cada corrida queda registrada con MLflow (`src/tracking.py`).

**Contenido** 1. Datos y verdad disponible · 2. El modelo · 3. Entrenamiento en la red real · 4. Qué aprendió ·
5. Un diagnóstico explicado · 6. Prueba con causa conocida (simulación) · 7. Resultados por fold · 8. Registro de
experimentos · 9. Conclusiones y limitaciones


## 1. Datos, grafo, incidentes y verdad disponible

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.ingesta import construir_dataset
from src.grafo import construir_grafo, grafo_sitios, jerarquia_sitios
from src.etiquetado import construir_incidentes, cargar_oro, cargar_reglas, etiqueta_plata
from src.evaluacion import (folds_temporales, rankear, evaluar, comparar, mcnemar, verdad_desde_simulacion)
from src.motor import HawkesTopologico
from src.sintetico import simular_avalanchas
from src.tracking import Corrida, huella_archivo, huella_df
from src.visual import vista_incidente, mostrar, guardar_html

BASE_DIR = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
REPORTES = ROOT / "reports" / "sprint2"
RES = ROOT / "results" / "motor_v1"; RES.mkdir(parents=True, exist_ok=True)
REGLAS = ROOT / "docs" / "reglas_causales_huawei.csv"
ORO = ROOT / "data" / "interim" / "oro_noc_completada.xlsx"

df = pd.read_parquet(PROC / "dataset_consolidado.parquet") if (PROC / "dataset_consolidado.parquet").exists() \
    else construir_dataset(str(BASE_DIR))
G = construir_grafo(str(BASE_DIR / "red.xlsx"), verbose=False)
J = jerarquia_sitios(G, grafo_sitios(G))
df_inc, incidentes = construir_incidentes(df, G)
folds = folds_temporales(df[["ventana_id", "ventana_ts"]], n_folds=4, min_entrenamiento=0.4, separacion=1,
                        pesos=incidentes.groupby("ventana_id").size())

# verdad de referencia: oro (NOC) si existe; si no, plata (manuales) si hay reglas; si no, ninguna
reglas = cargar_reglas(REGLAS)
verdad, tipo_verdad = None, "ninguna"
if ORO.exists():
    verdad, tipo_verdad = cargar_oro(ORO), "oro"
elif len(reglas) and len(incidentes):
    pl = etiqueta_plata(df_inc, reglas, G)
    verdad = (pl[pl["plata_estado"].isin(["unica", "desempate"])]
              .rename(columns={"plata_equipo": "causa_raiz_equipo", "plata_alarma": "causa_raiz_alarma"}))
    tipo_verdad = "plata"
    if verdad.empty:                     # hay reglas, pero ninguna aplica a los incidentes (p. ej. solo routers)
        verdad, tipo_verdad = None, "plata_sin_cobertura"
version_reglas = ",".join(sorted(reglas["version"].dropna().astype(str).unique())) if len(reglas) else "sin_reglas"
print(f"{len(incidentes):,} incidentes · {len(folds)} folds · verdad: {tipo_verdad}"
      + (f" ({len(verdad)} incidentes)" if verdad is not None else "") + f" · reglas: {version_reglas}")

> **Resumen.** Carga lo mismo que los notebooks 05–06 y decide con qué verdad se puede medir hoy: **oro** (planilla del NOC) si ya llegó; si no, **plata** (reglas de los manuales) si ya se cargaron; si no, ninguna. Sin verdad, el motor igual se entrena y diagnostica, pero sus aciertos solo se miden en la simulación (sección 6).

## 2. El modelo

Cada alarma $j$ de un incidente aparece **espontáneamente** (tasa de fondo $\mu$ de su tipo) o fue **disparada** por
una alarma anterior $i$ de un equipo cercano. La influencia de $i$ sobre $j$ es

$$\alpha_{\text{tipo}_i \to \text{tipo}_j}\;\cdot\;\beta e^{-\beta(t_j-t_i+\tau)}\;\cdot\;\gamma^{\,\text{saltos}(i,j)}$$

| Parámetro | Significado | Cómo se fija |
|---|---|---|
| $\alpha$ | cuánto un tipo de alarma dispara a otro | **se aprende** (EM, sin etiquetas) |
| $\beta$ | rapidez con que se apaga la influencia en el tiempo | se aprende |
| $\tau$ | tolerancia al desorden de llegada (*jitter*), 30 s | hiperparámetro |
| $\gamma$ | atenuación por salto físico en el grafo, 0,7 | hiperparámetro |
| dirección | una alarma solo dispara a equipos igual o más lejos del NOC (**aguas abajo**) | conocimiento topológico |

La **causa raíz** es la alarma con alta probabilidad de ser espontánea **y** que explica a muchas otras:
$\text{score} = P(\text{espontánea}) \times (1 + \text{alarmas que explica})$. Los hiperparámetros se optimizan en el
Sprint 3 (búsqueda de hiperparámetros).

## 3. Entrenamiento en la red real

En cada fold el motor se entrena solo con el pasado (ventanas de entrenamiento) y diagnostica los incidentes del bloque
de prueba. Si hay verdad, se mide; la corrida queda registrada con sus parámetros y la huella de los datos.

In [ ]:
HP = dict(tau=30.0, gamma=0.7, max_saltos=8, direccional=True)
huellas = {"huella_dataset": huella_df(df[["ventana_id", "NOMBRE DE EQUIPO EN RED", "alarmName"]]),
           "huella_red_xlsx": huella_archivo(BASE_DIR / "red.xlsx"),
           "huella_reglas": huella_archivo(REGLAS), "version_reglas": version_reglas, "verdad": tipo_verdad}
rankings_reales, por_fold_real, motor_final = [], [], None
with Corrida("motor_v1_red_real", params=HP, tags=huellas, raiz=ROOT) as run:
    for f in folds:
        tr = df_inc[df_inc["ventana_id"].isin(f["train"])]
        te = df_inc[df_inc["ventana_id"].isin(f["test"])]
        if tr.empty or te.empty:
            continue
        motor = HawkesTopologico(G, J=J, **HP).fit(tr)
        rk = motor.rankear(te).assign(fold=f["fold"])
        rankings_reales.append(rk)
        fila = {"fold": f["fold"], "incidentes_test": te["incidente_id"].nunique(),
                "iteraciones_em": len(motor.historial), "beta": motor.beta}
        if verdad is not None:
            res, _ = evaluar(rk, verdad[verdad["incidente_id"].isin(te["incidente_id"])])
            fila.update(res)
        por_fold_real.append(fila)
        run.metricas({k: v for k, v in fila.items() if k != "fold"}, paso=f["fold"])
        motor_final = motor
    if motor_final is not None:
        run.modelo(motor_final, "motor_v1_ultimo_fold")
por_fold_real = pd.DataFrame(por_fold_real)
rk_real = pd.concat(rankings_reales, ignore_index=True) if rankings_reales else pd.DataFrame()
display(por_fold_real)

> **Qué muestra la tabla.** Por fold: cuántos incidentes se diagnosticaron, cuántas iteraciones necesitó el EM para converger y la rapidez aprendida del núcleo (β, en 1/s: 1/β ≈ segundos que dura la influencia de una alarma). Si ya hay verdad, aparecen además Top-1/3/5, MRR, F1 y equipos revisados.

## 4. Qué aprendió el motor (interpretabilidad)

In [ ]:
if motor_final is not None:
    A = motor_final.matriz_alpha(min_alpha=0.05)
    display(A.head(15))
    if len(reglas):
        en_manual = set(zip(reglas["alarma_causa"], reglas["alarma_efecto"]))
        top = A.head(30)
        coincide = [(a, b) in en_manual for a, b in zip(top["tipo_causa"], top["tipo_efecto"])]
        print(f"De los 30 pares más fuertes aprendidos, {sum(coincide)} están en las reglas de los manuales.")

> **Qué muestra la tabla.** Los pares de tipos de alarma con mayor excitación aprendida (α): «cuando aparece A, suele disparar B». Es conocimiento extraído **solo de los datos**. Cuando estén cargadas las reglas de los manuales, la línea de abajo dice cuántos de esos pares también los afirma Huawei: una validación independiente del motor.

In [ ]:
if motor_final is not None and len(A):
    top = A.head(12)
    tipos = sorted(set(top["tipo_causa"]) | set(top["tipo_efecto"]))
    M = pd.DataFrame(0.0, index=tipos, columns=tipos)
    for a, b, v in top.itertuples(index=False):
        M.loc[a, b] = v
    fig, ax = plt.subplots(figsize=(7, 5.5))
    im = ax.imshow(M.values, cmap="Blues")
    ax.set_xticks(range(len(tipos)), tipos, rotation=60, ha="right", fontsize=8)
    ax.set_yticks(range(len(tipos)), tipos, fontsize=8)
    ax.set_xlabel("efecto"); ax.set_ylabel("causa"); fig.colorbar(im, ax=ax, label="α aprendido")
    ax.set_title("Quién dispara a quién (12 pares más fuertes)"); plt.tight_layout(); plt.show()

> **Qué muestra el gráfico.** La misma información como mapa de calor: filas = tipo que causa, columnas = tipo afectado. Celdas oscuras = relaciones fuertes aprendidas de los datos.

## 5. Un diagnóstico explicado

In [ ]:
if len(rk_real):
    INC = (incidentes[incidentes["incidente_id"].isin(rk_real["incidente_id"])]
           .sort_values("n_alarmas", ascending=False)["incidente_id"].iloc[0])
    display(motor_final.explicar(df_inc, INC, top=3))
    html = vista_incidente(G, df_inc[df_inc["incidente_id"] == INC], ranking=rk_real[rk_real["incidente_id"] == INC],
                           J=J, titulo=f"Diagnóstico del motor v1 · incidente {INC}")
    mostrar(html, alto=600)
    guardar_html(html, REPORTES / "diagnostico_motor_v1.html")

> **Qué muestra.** *(Tabla)* Las 3 causas que propone el motor para el incidente más grande de prueba: su probabilidad de ser espontánea, cuántas alarmas explica y cuáles explica directamente (con su probabilidad). Es la explicación que en el Sprint 5 redactará el LLM. *(Vista)* El incidente sobre la red con la posición `#k` del motor.

## 6. Prueba con causa conocida (simulación)

Mientras llegan las etiquetas, se mide el motor con avalanchas simuladas sobre la topología real: causa conocida,
retardos por salto, *jitter*, raíz que a veces llega tarde y ruido. Se compara contra los tres métodos de referencia y
una **ablación** (el mismo motor sin la restricción de dirección aguas abajo).

In [ ]:
df_sim, verdad_sim, _ = simular_avalanchas(G, n_ventanas=600, semilla=42)
inc_df_sim, inc_sim = construir_incidentes(df_sim, G)
V = verdad_desde_simulacion(inc_df_sim, verdad_sim)
folds_sim = folds_temporales(inc_df_sim[["ventana_id", "ventana_ts"]], n_folds=4)
metodos = {}
filas, detalles = [], {}
with Corrida("motor_v1_sintetico", params={**HP, "n_ventanas": 600, "semilla_sim": 42},
             tags={"verdad": "simulacion"}, raiz=ROOT) as run:
    for f in folds_sim:
        tr = inc_df_sim[inc_df_sim["ventana_id"].isin(f["train"])]
        te = inc_df_sim[inc_df_sim["ventana_id"].isin(f["test"])]
        vf = V[V["incidente_id"].isin(te["incidente_id"])]
        rks = {"T3 · Hawkes topológico": HawkesTopologico(G, J=J, **HP).fit(tr).rankear(te),
               "T3 sin dirección (ablación)": HawkesTopologico(G, J=J, **{**HP, "direccional": False}).fit(tr).rankear(te),
               "T0 · temporal": rankear(te, "temporal"),
               "T0′ · severidad": rankear(te, "severidad"),
               "T1 · topológico": rankear(te, "topologico", G=G)}
        for nombre, rk in rks.items():
            res, det = evaluar(rk, vf)
            filas.append({"fold": f["fold"], "metodo": nombre, **res})
            detalles.setdefault(nombre, []).append(det)
        hw = [x for x in filas if x["fold"] == f["fold"] and x["metodo"].startswith("T3 · ")][0]
        run.metricas({k: v for k, v in hw.items() if k not in ("fold", "metodo")}, paso=f["fold"])
    por_fold_sim = pd.DataFrame(filas)
    por_fold_sim.to_csv(RES / "por_fold_sintetico.csv", index=False)
    run.artefacto(RES / "por_fold_sintetico.csv")
display(por_fold_sim.pivot(index="metodo", columns="fold", values="top_1").round(3))

> **Qué muestra la tabla.** El Top-1 de cada método en cada fold de la simulación. Permite ver si una ventaja es estable o depende de un período.

## 7. Resultados por fold (media ± desviación)

In [ ]:
cols = ["top_1", "top_3", "top_5", "mrr", "f1", "equipos_revisados", "seg_por_incidente"]
resumen = por_fold_sim.groupby("metodo")[cols].agg(["mean", "std"])
tabla = pd.DataFrame({c: [f"{m:.3f} ± {s:.3f}" for m, s in zip(resumen[(c, 'mean')], resumen[(c, 'std')])] for c in cols},
                     index=resumen.index).sort_values("top_1", ascending=False)
display(tabla)
tabla.to_csv(RES / "resumen_por_fold_sintetico.csv")

> **Qué muestra la tabla.** Para cada método, la media ± desviación de las métricas sobre los 4 folds (DoD de la tarjeta *Resultados por fold*). `equipos_revisados` aproxima el tiempo de localización: cuántos equipos revisaría el operador siguiendo la lista hasta llegar a la causa.

In [ ]:
m = por_fold_sim.groupby("metodo")[["top_1", "top_3", "mrr"]].mean().sort_values("top_1")
ax = m.plot.barh(figsize=(8, 3.6), color=["#0B5D8F", "#2BA88A", "#F2A541"])
ax.set_xlim(0, 1); ax.set_xlabel("exactitud (media de 4 folds)"); ax.legend(frameon=False, loc="lower right")
ax.set_title("Motor v1 vs. métodos de referencia (simulación)"); plt.tight_layout(); plt.show()

> **Qué muestra el gráfico.** Top-1, Top-3 y MRR medios de cada método. La barra del motor (T3) frente a la mejor referencia es la mejora que aporta combinar tiempo, topología y lo aprendido.

In [ ]:
det_t3 = pd.concat(detalles["T3 · Hawkes topológico"])
mejor_ref = por_fold_sim[por_fold_sim["metodo"].str.startswith(("T0", "T1"))].groupby("metodo")["top_1"].mean().idxmax()
det_ref = pd.concat(detalles[mejor_ref])
b, c_, p = mcnemar(det_t3, det_ref)
print(f"McNemar T3 vs {mejor_ref}: solo acierta T3 = {b}, solo acierta la referencia = {c_}, "
      + ("p < 0,001" if p < 0.001 else f"p = {p:.3f}"))
import json
json.dump({"mejor_referencia": mejor_ref, "mcnemar": {"solo_T3": b, "solo_ref": c_, "p": p},
           "verdad": "simulacion", "hiperparametros": HP}, open(RES / "resumen_motor_v1.json", "w"), indent=2)

> **Qué muestra.** La prueba de McNemar compara el motor con la mejor referencia sobre los **mismos** incidentes: p < 0,05 indica que la diferencia no es casualidad.

## 8. Registro de experimentos (MLflow)

Cada corrida de este notebook quedó registrada con sus hiperparámetros, la huella de los datos (`huella_dataset`,
`huella_red_xlsx`), la versión de las reglas, las métricas por fold y el modelo entrenado. Para verlas y compararlas,
desde la carpeta del repositorio:

```
mlflow ui --backend-store-uri sqlite:///mlruns/mlflow.db
```
y abrir http://127.0.0.1:5000. Sin MLflow instalado, el mismo registro queda en `results/experimentos.jsonl`.

In [ ]:
import json
for exp in ["motor_v1_red_real", "motor_v1_sintetico"]:
    r = json.load(open(ROOT / "results" / exp / "ultima_corrida.json"))
    print(f"{exp}: {r['estado']} · {r['duracion_s']} s · MLflow run {r['mlflow_run_id']} · "
          f"{len(r['metricas'])} registros de métricas")

> **Resumen.** Confirma que las dos corridas quedaron registradas (estado, duración e identificador en MLflow).

## 9. Conclusiones y limitaciones

- El motor v1 se entrena **sin etiquetas** y produce un ranking Top-k explicable por incidente.
- En la simulación supera a los métodos de referencia; la ablación muestra cuánto aporta la dirección aguas abajo.
- **Limitación:** la simulación propaga las fallas solo aguas abajo, igual que la restricción del motor, así que esa
  ventaja es optimista. La medición válida para la tesis será con la etiqueta **oro** (NOC) y, por segmento, con la
  **plata** (manuales: v1 = RTN 320; routers pendientes).
- Siguiente (Sprint 3): búsqueda de hiperparámetros (τ, γ, saltos), control de overfitting y variante GNN opcional.